# 第 1 周末练习 —— 技术问答解释器（OpenAI + Ollama）

## 练习目标（理念）

为了展示你对 **OpenAI API** 以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一个技术问题（例如「这段 Python 代码在干什么？」）
- **输出**：清晰、分步的解释（Markdown）
- **额外要求**：用**流式（streaming）**边生成边用 `update_display` 刷新，而不是等整段答完才一次性显示

这是你在课程期间自己也能天天用的工具：遇到看不懂的代码，丢进来问模型。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `chat.completions.create(..., stream=True)` |
| `messages`（system / user） | system 定「怎么答」，user 由 `gen_user_prompt` 拼出 |
| 流式输出 | 累加 `delta.content`，配合 `update_display` |
| OpenAI 云端模型 | `gpt-5-nano`（常量 `MODEL_GPT`） |
| Ollama 本地模型 | `llama3.2`（常量 `MODEL_LLAMA`），OpenAI 兼容 `/v1` |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好上级目录的 `.env`（本笔记本用 `Path("../../../.env")`）：至少有 `OPENAI_API_KEY`
3. 若要跑 Llama：本机启动 Ollama，并已 `ollama pull llama3.2`
4. 在「提问」单元格改写 `question`，再分别调用 GPT / Llama 函数对比


In [76]:
# ========== 导入：路径、环境、爬虫辅助、OpenAI、笔记本展示 ==========

# 导入标准库 os：拼路径、读环境变量
import os
# 导入标准库 sys：把上级目录塞进 module 搜索路径，便于 import scraper
import sys
# 导入 requests：本练习主要用于可选的本地 Ollama 连通性探测（见后面注释掉的 get）
import requests

# 把父目录加入 sys.path，这样可以 import 同级/上级的 scraper 模块
sys.path.append(os.path.abspath(".."))
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量
from dotenv import load_dotenv
# 从 scraper 导入网页抓取辅助（本练习问答主路径未必用到，但保留原导入）
from scraper import fetch_website_links, fetch_website_contents
# 从 openai 导入 OpenAI 客户端类：云端 GPT 与 Ollama 兼容端都用它
from openai import OpenAI
# 从 pathlib 导入 Path：用面向对象路径指向 ../../../.env
from pathlib import Path
# 从 IPython.display 导入展示工具：流式时用 display + update_display 刷新同一块 Markdown
from IPython.display import Markdown, display, update_display


In [65]:
# ========== 常量：模型名字集中写在一处，后面只改这里 ==========

# OpenAI 云端模型 id（字符串必须与账号可用模型一致）
MODEL_GPT = 'gpt-5-nano'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2
MODEL_LLAMA = 'llama3.2'


In [87]:
# ========== 环境检查：加载 .env、校验 OPENAI_API_KEY、创建客户端 ==========

# 从相对本笔记本的路径加载上级仓库里的 .env（三层 .. 再进 .env）
load_dotenv(Path("../../../.env"))
# 读取 OpenAI 密钥（常见名 OPENAI_API_KEY）
api_key = os.getenv("OPENAI_API_KEY")

# 没有密钥：提示去排查笔记本（错误文案保持英文原文，勿改译）
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
# 有密钥但前缀不像 sk-proj-：可能拿错了 key
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
# 首尾有空白：常见复制粘贴问题
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    # 看起来格式正常
    print("API key found and looks good so far!")
# 创建默认 OpenAI 客户端（会从环境变量读 OPENAI_API_KEY）
openai = OpenAI()


API key found and looks good so far!


In [88]:
# ========== system prompt：定「编程导师」角色与输出格式 ==========

# 发给模型的系统指令保留英文；改译会改变回答风格/行为
system_prompt = """
You are a knowledgeable programming tutor.

Your job is to help students understand technical questions and code.
Always provide clear, step-by-step explanations.

Respond in Markdown format.
Do NOT wrap the entire response in a Markdown code block.
"""


In [43]:
# ========== 组装 user prompt：把具体问题嵌进固定英文模板 ==========

def gen_user_prompt(question):
    # f-string：把调用方传入的 question 插进模板；模板正文保持英文
    user_prompt = f"""
Here is the technical question I need help with:

{question}

Please explain it clearly and step-by-step.
"""
    # 返回完整 user 消息字符串，供 messages 列表使用
    return user_prompt


In [48]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# 示例问题：请解释生成器函数 count_up_to；发给模型的内容保持英文/代码原样
question = """
Please explain what this code does and why:

def count_up_to(n):
    i = 1
    while i <= n:
        yield i
        i += 1
"""


In [66]:
# ========== GPT 流式回答：边收 delta 边刷新同一块 Markdown ==========

def answer_question_gpt(question):
    # chat.completions.create + stream=True：返回可迭代的流式事件
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages = [
            # system：角色与格式规则
            {"role":"system", "content": system_prompt},
            # user：由 gen_user_prompt 把 question 包进模板
            {"role":"user", "content": gen_user_prompt(question)}
        ],
        stream=True
    )

    # response：累加到目前为止的完整文本
    response = ""
    # display_id=True：拿到可更新的 display handle，避免每次新开一块输出
    display_handle = display(Markdown(""), display_id=True)
    # 遍历流式 chunk
    for chunk in stream:
        # delta.content 可能是 None（例如结束标记）；用 or '' 变成空串再拼接
        response += chunk.choices[0].delta.content or ''
        # 用同一 display_id 刷新 Markdown，实现「打字机」效果
        update_display(Markdown(response), display_id=display_handle.display_id)


In [84]:
# ========== Ollama：用 OpenAI 兼容的 /v1 端点创建第二个客户端 ==========

# 可选连通性探测（已注释）：GET 根地址确认本机 11434 是否在听
# requests.get("http://localhost:11434").content

# Ollama 的 OpenAI 兼容基址：注意带 /v1，供 chat.completions 使用
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# api_key 对本地 Ollama 通常任意非空即可；这里沿用原字符串 'ollama'
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [85]:
# ========== Llama 流式回答：接口形状与 GPT 相同，只是换客户端/模型 ==========

def answer_question_llama(question):
    # 走本地 ollama 客户端；model 用 MODEL_LLAMA
    stream = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages = [
            {"role":"system", "content": system_prompt},
            {"role":"user", "content": gen_user_prompt(question)}
        ],
        stream=True
    )

    # 同样累加文本 + update_display 刷新
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [86]:
# ========== 实际调用：用上面的 question 问本地 Llama ==========

# 传入全局 question；需本机 Ollama 已运行且已拉取 llama3.2
answer_question_llama(question)


# Understanding the `count_up_to` Function
=====================================

The `count_up_to` function is a generator that yields numbers from 1 up to, but not including, the given number `n`.

### Step-by-Step Explanation
-------------------------

Here's how it works:

1. **Initialization**: We initialize a counter variable `i` to 1.
2. **Loop Condition**: The loop condition checks whether `i` is less than or equal to `n`. As long as this condition is true, the loop will continue.
3. **Yielding a Value**: Inside the loop, we use the `yield` keyword to produce a value, which in this case is the current value of `i`.
4. **Incrementing `i`**: After yielding a value, we increment `i` by 1 to prepare it for the next iteration.

### Key Points
------------

*   The `count_up_to` function returns an iterator that produces numbers on-the-fly, rather than all at once.
*   It does not store any of these values in memory; instead, it generates them as needed when requested.
*   Once the loop condition is no longer true (i.e., after `n` has been reached), there are no more values to yield.

### Example Use Case
-------------------

Here's an example of how you might use this function:
```python
for num in count_up_to(5):
    print(num)
```

Output:
```
1
2
3
4
5
```
As the loop iterates and yields numbers, it skips over any value that would cause the condition (`i <= n`) to become false.